# 02 - Audio preprocessing and MFCC feature extraction

This notebook creates the `silence` class from background-noise recordings, checks the complete class/split distribution, standardizes clips to one-second mono audio at 16 kHz, and extracts MFCC summary features for SVM and Random Forest experiments.

**Run notebook 01 first.** The downloaded dataset and generated index remain local and are intentionally excluded from GitHub.

## 1. Project paths and imports

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook is stored in notebooks/, while src/ is at the repository root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    build_audio_index,
    save_audio_index,
    class_distribution,
    extract_mfcc_features,
    split_feature_arrays,
)

NOTEBOOK_DATA = Path.cwd() / "data" if Path.cwd().name == "notebooks" else PROJECT_ROOT / "notebooks" / "data"
INDEX_PATH = NOTEBOOK_DATA / "processed" / "audio_index.csv"
DATASET_DIR = NOTEBOOK_DATA / "raw" / "speech_commands_v0.02"
PROCESSED_DIR = NOTEBOOK_DATA / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
DURATION_SECONDS = 1.0
N_MFCC = 40

print("Repository root:", PROJECT_ROOT)
print("Speech index:", INDEX_PATH)
print("Extracted dataset:", DATASET_DIR)

## 2. Build the complete index, including silence

In [ ]:
audio_index = build_audio_index(
    INDEX_PATH,
    DATASET_DIR,
    silence_stride_seconds=1.0,
)

complete_index_path = save_audio_index(
    audio_index,
    PROCESSED_DIR / "audio_index_with_silence.csv",
)

print(f"Total indexed clips: {len(audio_index):,}")
print("Target classes:", sorted(audio_index["target"].astype(str).unique()))
print("Saved complete index:", complete_index_path)

## 3. Inspect class distribution and official splits

In [ ]:
distribution = class_distribution(audio_index)
display(distribution)

split_totals = audio_index.groupby("split", observed=True).size().rename("recordings")
display(split_totals.to_frame())

ax = distribution.T.plot(kind="bar", figsize=(14, 6))
ax.set_title("Class distribution after adding silence")
ax.set_xlabel("Target class")
ax.set_ylabel("Number of audio clips")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

assert set(audio_index["target"].astype(str).unique()) == {
    "yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go", "unknown", "silence"
}, "The index should contain exactly the 12 planned target classes."
assert set(audio_index["split"].astype(str).unique()) == {"train", "validation", "test"}
print("12 target classes and all three data splits are present.")

## 4. Verify silence split isolation

Every background recording must belong to exactly one split. This prevents adjacent segments from the same continuous noise recording appearing in both training and evaluation data.

In [ ]:
silence_index = audio_index[audio_index["target"].astype(str) == "silence"].copy()
silence_source_splits = silence_index.groupby("speaker_id", observed=True)["split"].nunique()
assert (silence_source_splits == 1).all(), "A background recording appears in more than one split."
display(
    silence_index.groupby(["split", "speaker_id"], observed=True)
    .size()
    .rename("one_second_segments")
    .reset_index()
)
print("Background recording split-isolation check passed.")

## 5. Test MFCC extraction on a small sample

This quick check extracts at most five clips per class per split. It verifies the feature shape and label alignment without processing the entire dataset.

In [ ]:
X_preview, y_preview, preview_metadata = extract_mfcc_features(
    audio_index,
    sample_rate=SAMPLE_RATE,
    duration_seconds=DURATION_SECONDS,
    n_mfcc=N_MFCC,
    max_samples_per_class=5,
    random_state=42,
)

print("Preview feature matrix:", X_preview.shape)
print("Preview labels:", pd.Series(y_preview).value_counts().sort_index().to_dict())
assert X_preview.shape[1] == N_MFCC * 2
assert len(X_preview) == len(y_preview) == len(preview_metadata)
assert np.isfinite(X_preview).all(), "MFCC features contain NaN or infinite values."
print("MFCC preview checks passed.")

## 6. Extract MFCC features for the full dataset

This cell processes every indexed recording. It can take several minutes depending on your CPU. The output is a compact feature matrix: 40 MFCC means + 40 MFCC standard deviations per clip. Scaling and model fitting will be handled inside the model-training pipelines to avoid data leakage.

In [ ]:
X, y, feature_metadata = extract_mfcc_features(
    audio_index,
    sample_rate=SAMPLE_RATE,
    duration_seconds=DURATION_SECONDS,
    n_mfcc=N_MFCC,
    max_samples_per_class=None,
    show_progress=True,
)

feature_splits = split_feature_arrays(X, y, feature_metadata)
for split_name, arrays in feature_splits.items():
    print(f"{split_name:10s} X={arrays['X'].shape}, y={arrays['y'].shape}")

assert np.isfinite(X).all(), "Full feature matrix contains NaN or infinite values."
assert X.shape[1] == N_MFCC * 2

## 7. Save the feature arrays locally

In [ ]:
feature_output = PROCESSED_DIR / "mfcc_features.npz"
np.savez_compressed(
    feature_output,
    X_train=feature_splits["train"]["X"],
    y_train=feature_splits["train"]["y"],
    X_validation=feature_splits["validation"]["X"],
    y_validation=feature_splits["validation"]["y"],
    X_test=feature_splits["test"]["X"],
    y_test=feature_splits["test"]["y"],
    feature_names=np.array(
        [f"mfcc_{i+1}_{stat}" for stat in ("mean", "std") for i in range(N_MFCC)]
    ),
)
print("Saved feature arrays:", feature_output.resolve())
print("Feature matrix shape:", X.shape)
print("Preprocessing milestone complete.")

## Interpretation and next step

- The dataset now has 12 target classes, including silence clips sampled from background recordings.
- Official speech-command train, validation, and test assignments are preserved.
- Background-noise source files are isolated by split to reduce leakage.
- Each recording is represented by 80 MFCC summary features (40 means and 40 standard deviations).
- The feature arrays are saved locally and should not be committed to GitHub.

The next notebook will compare classical classifiers using training-only preprocessing, validation-based model selection, and a final untouched test evaluation.